# Revenue Dashboard

Lavanya Ginjupalli

Created:       May 23, 2026

Last Updated:  May 23, 2026

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path
import dash
from dash import dcc, html, Input, Output
import webbrowser

# =========================================================
# LOAD DATA
# =========================================================
base = Path("../data/gold")

df_country = pd.read_csv(base / "country_revenue.csv")
df_product = pd.read_csv(base / "product_revenue.csv")
df_monthly = pd.read_csv(base / "monthly_revenue.csv")
df_weekly = pd.read_csv(base / "weekly_revenue.csv")

# =========================================================
# CLEAN COLUMN NAMES
# =========================================================
for df in [df_country, df_product, df_monthly, df_weekly]:
    df.columns = df.columns.str.strip()

# =========================================================
# HELPER FUNCTION
# =========================================================
def find_col(df, keywords):
    for c in df.columns:
        if any(k in c.lower() for k in keywords):
            return c
    return None

country_col_country = "Country"
country_col_revenue = find_col(df_country, ["revenue", "price", "total"])

prod_col_category = find_col(df_product, ["category", "product"])
prod_col_revenue = find_col(df_product, ["revenue", "price", "total"])

# =========================================================
# MONTHLY DATA PREP
# =========================================================
df_monthly["Date"] = pd.to_datetime(
    df_monthly["Year"].astype(str) + "-" +
    df_monthly["Month"].astype(str) + "-01",
    errors="coerce"
)

df_monthly = df_monthly.dropna(subset=["Date"])
df_monthly = df_monthly.sort_values("Date")

df_monthly["MonthName"] = df_monthly["Date"].dt.strftime("%b")

# =========================================================
# NUMERIC CLEAN
# =========================================================
def safe_numeric(df, col):
    if col and col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

safe_numeric(df_country, country_col_revenue)
safe_numeric(df_product, prod_col_revenue)
safe_numeric(df_product, "TotalRevenue")
safe_numeric(df_product, "TotalQuantity")
safe_numeric(df_monthly, "TotalPrice")
safe_numeric(df_weekly, "TotalPrice")

# =========================================================
# DASH APP
# =========================================================
app = dash.Dash(__name__)
server = app.server

# =========================================================
# FILTER OPTIONS
# =========================================================
country_options = [
    {"label": c, "value": c}
    for c in sorted(df_country[country_col_country].dropna().unique())
]

category_options = [
    {"label": c, "value": c}
    for c in sorted(df_product[prod_col_category].dropna().unique())
]

# =========================================================
# STYLES
# =========================================================
page_style = {
    "backgroundColor": "#0B1020",
    "minHeight": "100vh",
    "paddingBottom": "30px",
    "fontFamily": "Segoe UI"
}

card_style = {
    "backgroundColor": "#131C31",
    "borderRadius": "18px",
    "padding": "15px",
    "boxShadow": "0px 6px 18px rgba(0,0,0,0.35)"
}

chart_style = {
    "backgroundColor": "#131C31",
    "borderRadius": "18px",
    "padding": "15px",
    "boxShadow": "0px 6px 18px rgba(0,0,0,0.35)"
}

filter_label = {
    "color": "white",
    "fontWeight": "600",
    "marginBottom": "5px"
}

# =========================================================
# KPI STYLES
# =========================================================
kpi_common = {
    "padding": "20px",
    "borderRadius": "18px",
    "color": "white",
    "textAlign": "center",
    "boxShadow": "0px 6px 18px rgba(0,0,0,0.3)"
}

kpi1 = {**kpi_common, "background": "linear-gradient(135deg,#4facfe,#00f2fe)"}
kpi2 = {**kpi_common, "background": "linear-gradient(135deg,#43e97b,#38f9d7)"}
kpi3 = {**kpi_common, "background": "linear-gradient(135deg,#fa709a,#fee140)"}
kpi4 = {**kpi_common, "background": "linear-gradient(135deg,#667eea,#764ba2)"}

# =========================================================
# LAYOUT
# =========================================================
app.layout = html.Div(style=page_style, children=[

    # HEADER
    html.Div([
        html.H1(
            "📊 Executive Revenue Dashboard",
            style={
                "textAlign": "center",
                "color": "white",
                "padding": "20px",
                "marginBottom": "10px"
            }
        )
    ]),

    # FILTER SECTION
    html.Div(style={
        "width": "92%",
        "margin": "auto"
    }, children=[

        html.Div(style=card_style, children=[

            html.Div(style={
                "display": "grid",
                "gridTemplateColumns": "1fr 1fr 1fr 1fr",
                "gap": "15px"
            }, children=[

                # COUNTRY
                html.Div([
                    html.Label("🌍 Country Filter", style=filter_label),
                    dcc.Dropdown(
                        id="country",
                        options=country_options,
                        multi=True,
                        placeholder="Select Country"
                    )
                ]),

                # CATEGORY
                html.Div([
                    html.Label("📦 Product Category", style=filter_label),
                    dcc.Dropdown(
                        id="category",
                        options=category_options,
                        multi=True,
                        placeholder="Select Category"
                    )
                ]),

                # MONTH / WEEK
                html.Div([
                    html.Label("📅 Revenue Trend Type", style=filter_label),
                    dcc.Dropdown(
                        id="time_type",
                        options=[
                            {"label": "Monthly Revenue", "value": "month"},
                            {"label": "Weekly Revenue", "value": "week"}
                        ],
                        value="month",
                        clearable=False
                    )
                ]),

                # TOP N
                html.Div([
                    html.Label("🏆 Top Products Filter", style=filter_label),
                    dcc.Dropdown(
                        id="top_n",
                        options=[
                            {"label": "Top 5 Products", "value": 5},
                            {"label": "Top 10 Products", "value": 10},
                            {"label": "Top 20 Products", "value": 20},
                        ],
                        value=10,
                        clearable=False
                    )
                ]),

            ])
        ])
    ]),

    # KPI ROW
    html.Div(style={
        "display": "grid",
        "gridTemplateColumns": "repeat(4, 1fr)",
        "gap": "18px",
        "width": "92%",
        "margin": "25px auto"
    }, children=[

        html.Div(style=kpi1, children=[
            html.H4("💰 Total Revenue"),
            html.H2(f"${df_country[country_col_revenue].sum():,.0f}")
        ]),

        html.Div(style=kpi2, children=[
            html.H4("📦 Products"),
            html.H2(len(df_product))
        ]),

        html.Div(style=kpi3, children=[
            html.H4("🛒 Quantity Sold"),
            html.H2(f"{df_product['TotalQuantity'].sum():,.0f}")
        ]),

        html.Div(style=kpi4, children=[
            html.H4("🌎 Countries"),
            html.H2(df_country[country_col_country].nunique())
        ]),
    ]),

    # CHART GRID
    html.Div(style={
        "display": "grid",
        "gridTemplateColumns": "1fr 1fr",
        "gap": "20px",
        "width": "92%",
        "margin": "auto"
    }, children=[

        # MAP
        html.Div(style=chart_style, children=[
            html.H3("🌍 Revenue by Country", style={"color": "white"}),
            dcc.Graph(id="map", style={"height": "430px"})
        ]),

        # CATEGORY
        html.Div(style=chart_style, children=[
            html.H3("📦 Revenue by Product Category", style={"color": "white"}),
            dcc.Graph(id="product", style={"height": "430px"})
        ]),

        # TOP PRODUCTS
        html.Div(style=chart_style, children=[
            html.H3("🏆 Top Products (Revenue vs Quantity)", style={"color": "white"}),
            dcc.Graph(id="top_products", style={"height": "430px"})
        ]),

        # TREND
        html.Div(style=chart_style, children=[
            html.H3("📈 Revenue Trend", style={"color": "white"}),
            dcc.Graph(id="trend_chart", style={"height": "430px"})
        ]),
    ])
])

# =========================================================
# CALLBACK
# =========================================================
@app.callback(
    Output("map", "figure"),
    Output("product", "figure"),
    Output("top_products", "figure"),
    Output("trend_chart", "figure"),
    Input("country", "value"),
    Input("category", "value"),
    Input("time_type", "value"),
    Input("top_n", "value"),
)
def update_dashboard(countries, categories, time_type, top_n):

    dfc = df_country.copy()
    dfp = df_product.copy()

    # FILTERS
    if countries:
        dfc = dfc[dfc[country_col_country].isin(countries)]

    if categories:
        dfp = dfp[dfp[prod_col_category].isin(categories)]

    # =====================================================
    # FLOATING WORLD MAP
    # =====================================================
    df_map = dfc.groupby(
        country_col_country,
        as_index=False
    )[country_col_revenue].sum()

    fig_map = px.choropleth(
        df_map,
        locations=country_col_country,
        locationmode="country names",
        color=country_col_revenue,
        color_continuous_scale=[
         "#f1a6f1",
         "#d6fa52",
         "#1BC9C0",
         "#73ec10"
]
    )

    fig_map.update_geos(
        showframe=False,
        showcoastlines=False,
        showcountries=False,
        projection_type="natural earth",
        bgcolor="rgba(0,0,0,0)",
        showland=True,
        landcolor="#EAF2FF"
    )

    fig_map.update_layout(
        paper_bgcolor="#131C31",
        plot_bgcolor="#131C31",
        margin=dict(l=0, r=0, t=0, b=0),
        font_color="white"
    )

    # =====================================================
    # PRODUCT CATEGORY
    # =====================================================
    df_cat = dfp.groupby(
        prod_col_category,
        as_index=False
    )[prod_col_revenue].sum()

    fig_product = px.pie(
        df_cat,
        names=prod_col_category,
        values=prod_col_revenue,
        hole=0.4
    )

    fig_product.update_layout(
        paper_bgcolor="#131C31",
        font_color="white"
    )

    # =====================================================
    # TOP PRODUCTS
    # =====================================================
    df_top = dfp.groupby(
        ["StockCode", "Description", prod_col_category],
        as_index=False
    ).agg({
        "TotalRevenue": "sum",
        "TotalQuantity": "sum"
    })

    df_top = df_top.sort_values(
        "TotalRevenue",
        ascending=False
    ).head(top_n)

    fig_top = px.bar(
        df_top,
        x="Description",
        y=["TotalRevenue", "TotalQuantity"],
        barmode="group"
    )

    fig_top.update_layout(
        paper_bgcolor="#131C31",
        plot_bgcolor="#131C31",
        font_color="white",
        xaxis_tickangle=-45
    )

    # =====================================================
    # MONTHLY / WEEKLY TREND
    # =====================================================
    if time_type == "week":

        dfw = df_weekly.groupby(
            "Week",
            as_index=False
        )["TotalPrice"].sum()

        fig_trend = px.line(
            dfw,
            x="Week",
            y="TotalPrice",
            markers=True,
            title="Weekly Revenue Trend"
        )

    else:

        dfm = df_monthly.groupby(
            ["Month", "MonthName"],
            as_index=False
        )["TotalPrice"].sum()

        dfm = dfm.sort_values("Month")

        fig_trend = px.line(
            dfm,
            x="MonthName",
            y="TotalPrice",
            markers=True,
            title="Monthly Revenue Trend"
        )

    fig_trend.update_layout(
        paper_bgcolor="#131C31",
        plot_bgcolor="#131C31",
        font_color="white"
    )

    return fig_map, fig_product, fig_top, fig_trend

# =========================================================
# RUN APP
# =========================================================
if __name__ == "__main__":
    webbrowser.open("http://127.0.0.1:8080")
    app.run(host="127.0.0.1", port=8080, debug=False)

C:\Users\lginj\AppData\Local\Temp\ipykernel_16492\3685315012.py:317: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig_map = px.choropleth(
C:\Users\lginj\AppData\Local\Temp\ipykernel_16492\3685315012.py:317: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig_map = px.choropleth(
